In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = '/content/drive/My Drive/Colab Notebooks/PriceIt/data/listings.csv.gz'
df = pd.read_csv(file_path)

In [4]:
# clean price column
df["price"] = (
    df["price"]
    .str.replace(r"[\$,]", "", regex=True)
    .astype(float)
)

df = df.dropna(subset="price")

# remove extreme outliers
upper_limit = df["price"].quantile(0.95) # pandas series method
df = df[df["price"] <= upper_limit]

# log-transform prices
df["log_price"] = np.log1p(df["price"])

In [5]:
# clean missing bathroom values using bathroom_text values
def parse_bathrooms(text):
  if pd.isna(text):
    return np.nan
  text_lower = text.lower()
  if "half" in text_lower:
    return 0.5
  match = re.search(r"[\d\.]+", text)
  if match:
    return float(match.group())
  return np.nan

df["bathrooms_imputed"] = df["bathrooms"].fillna(df["bathrooms_text"].apply(parse_bathrooms))
df["bathrooms"] = df["bathrooms_imputed"].fillna(df["bathrooms_imputed"].median()) # fill misisng values with median
df.drop(columns="bathrooms_imputed", inplace=True)

In [6]:
# clean missing bedroom values
df = df.dropna(subset=["bedrooms"]) # 6 observations

In [7]:
# one-hot encoding for neighbourhood_cleansed values
df = pd.get_dummies(df, columns=["neighbourhood_cleansed"], drop_first=True) # 76 neighborhoods

In [8]:
# encode instant_bookable as binary
df["instant_bookable_encoded"] = df["instant_bookable"].map({"t": 1, "f": 0})

In [9]:
# one-hot encoding for room_type values
df = pd.get_dummies(df, columns=["room_type"], drop_first=True)

In [10]:
df["amenities_count"] = df["amenities"].apply(lambda x: len(x.split(",")))

In [11]:
df["guests_per_bedroom"] = df["accommodates"] / df["bedrooms"].replace(0, 1)

In [12]:
# Find top 10 most common property types
top_types = df["property_type"].value_counts().nlargest(10).index

# Replace rare types with "Other"
df["property_type_cleaned"] = df["property_type"].apply(lambda x: x if x in top_types else "Other")

df = pd.get_dummies(df, columns=["property_type_cleaned"], drop_first=True)

In [13]:
df["bathrooms_per_guest"] = df["bathrooms"] / df["accommodates"].replace(0, 1)

In [14]:
df["guests_per_night"] = df["accommodates"] / df["minimum_nights"].replace(0, 1)

In [15]:
# Step 1: Define your top amenities (including city skyline view)
top_amenities = [
    "City skyline view",
    "Pool",
    "Hot tub",
    "Air conditioning",
    "Washer",
    "Dryer",
    "Gym",
    "Balcony",
    "Kitchen",
    "Wifi",
    "Lake access"
]

for amenity in top_amenities:
    col_name = amenity.lower().replace(" ", "_")
    df[col_name] = df["amenities"].str.contains(amenity, case=False, na=False).astype(int)

In [16]:
amenity_cols = [amenity.lower().replace(" ", "_") for amenity in top_amenities]
numeric_features = ["bedrooms", "bathrooms", "accommodates", "amenities_count", "guests_per_bedroom", "minimum_nights", "bathrooms_per_guest", "guests_per_night"] + amenity_cols
categorical_features = [col for col in df.columns if "property_type_" in col or "neighbourhood_cleansed_" in col] + ["instant_bookable_encoded"]

X = df[numeric_features + categorical_features]
y = df["log_price"]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [19]:
y_pred_log = model.predict(X_test)

# metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2 = r2_score(y_test, y_pred_log)

print(f"RMSE (log scale): {rmse:.2f}")
print(f"R2 (log scale): {r2:.3f}")

RMSE (log scale): 0.38
R2 (log scale): 0.654


In [20]:
model.score(X_train, y_train)

0.6821634253338124

In [21]:
model.score(X_test, y_test)

0.6538853339807607

In [22]:
# predicted prices in original dollars
y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

# RMSE in dollars (interpretive, not comparable to R²)
rmse_dollars = np.sqrt(mean_squared_error(y_test_original, y_pred))
print(f"RMSE in dollars (interpretive): ${rmse_dollars:.2f}")

RMSE in dollars (interpretive): $72.03


In [23]:
comparison_df = pd.DataFrame({
    "Actual Price": y_test_original,
    "Predicted Price": y_pred
})

comparison_df["Error"] = comparison_df["Predicted Price"] - comparison_df["Actual Price"]
comparison_df["Absolute Error"] = comparison_df["Error"].abs()

comparison_df.head(10)

,Actual Price,Predicted Price,Error,Absolute Error
728,89.0,81.054926,-7.945074,7.945074
3230,141.0,176.732258,35.732258,35.732258
120,149.0,162.258598,13.258598,13.258598
5573,162.0,174.540124,12.540124,12.540124
2339,121.0,151.693359,30.693359,30.693359
4491,80.0,68.987068,-11.012932,11.012932
4763,161.0,143.765810,-17.234190,17.234190
7145,28.0,60.616405,32.616405,32.616405
2077,68.0,74.441713,6.441713,6.441713
3310,232.0,208.657373,-23.342627,23.342627


In [24]:
model2 = RandomForestRegressor()

model2.fit(X_train, y_train)

RandomForestRegressor()

In [25]:
model2.score(X_test, y_test)

0.7280859059141922

In [29]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "min_samples_split": [2, 4, 6, 8],
    "max_depth": [None, 4, 8]
    }

model2 = RandomForestRegressor()

grid_search = GridSearchCV(model2, param_grid, cv=5,
                           scoring="neg_mean_squared_error",
                           return_train_score=True)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(),
             param_grid={'max_depth': [None, 4, 8],
                         'min_samples_split': [2, 4, 6, 8],
                         'n_estimators': [100, 200, 300]},
             return_train_score=True, scoring='neg_mean_squared_error')

In [30]:
best_model2 = grid_search.best_estimator_

In [31]:
best_model2.score(X_test, y_test)

0.7493222792907495

In [26]:
# Train
model_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model_xgb.fit(X_train, y_train)

# Evaluate
y_pred_train = model_xgb.predict(X_train)
y_pred_test = model_xgb.predict(X_test)

print("Train R²: {:.4f}".format(r2_score(y_train, y_pred_train)))
print("Test R²:  {:.4f}".format(r2_score(y_test, y_pred_test)))

print("Train RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_train, y_pred_train))))
print("Test RMSE:  {:.4f}".format(np.sqrt(mean_squared_error(y_test, y_pred_test))))

Train R²: 0.8512
Test R²:  0.7394
Train RMSE: 0.2528
Test RMSE:  0.3311


In [27]:
# RMSE in original dollar scale
rmse_dollars = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred_test)))
print("Test RMSE in dollars: ${:.2f}".format(rmse_dollars))

Test RMSE in dollars: $62.27


In [28]:
# Average dollar error
dollar_errors = np.abs(np.expm1(y_pred_test) - np.expm1(y_test))
print("Mean Absolute Error: ${:.2f}".format(dollar_errors.mean()))
print("Median Absolute Error: ${:.2f}".format(np.median(dollar_errors)))

Mean Absolute Error: $39.92
Median Absolute Error: $24.18


In [30]:
mape = np.mean(np.abs(np.expm1(y_pred_test) - np.expm1(y_test)) / np.expm1(y_test)) * 100
print("Mean Absolute Percentage Error: {:.1f}%".format(mape))

Mean Absolute Percentage Error: 25.3%


In [29]:
import joblib

# Save your trained XGBoost model
joblib.dump(model_xgb, "priceit_model.pkl")

# Save the exact feature columns your model was trained on
joblib.dump(list(X.columns), "feature_columns.pkl")

# Save neighborhood and property type lists for the frontend dropdowns
neighborhoods = sorted([col.replace("neighbourhood_cleansed_", "") for col in X.columns if "neighbourhood_cleansed_" in col])
property_types = sorted([col.replace("property_type_cleaned_", "") for col in X.columns if "property_type_" in col])

joblib.dump(neighborhoods, "neighborhoods.pkl")
joblib.dump(property_types, "property_types.pkl")

print("Saved successfully")
print("Neighborhoods:", len(neighborhoods))
print("Property types:", len(property_types))

Saved successfully
Neighborhoods: 75
Property types: 10
